# Módulo 1 · 01 — Datos y construcción de las series

Conjunto: *Operación de pasajeros y despacho de vehículos en la modalidad de transporte de pasajeros por carretera* (Superintendencia de Transporte), [datos.gov.co/d/eh75-8ah6](https://www.datos.gov.co/d/eh75-8ah6). ~49 M registros: la agregación se hace en el servidor con SoQL.

Salidas (versionadas en `data/processed/`): `series_diarias.parquet`, `calendario.parquet`, `rutas.csv`, `calidad.csv`.

In [ ]:
# parameters
DEMO = False
FORZAR_DESCARGA = False

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "modulo1_demanda"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla, semillas_experimento
from common.config import cargar_config
import m1lib as m1

fijar_semilla()
cfg = cargar_config("modulo1")
M1 = RAIZ / "modulo1_demanda"
DATA, FIG, MOD = M1 / "data", M1 / "figures", M1 / "models"
H, TAU = cfg["pronostico"]["horizonte"], cfg["pronostico"]["cuantil_alto"]
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Caso de uso

_TODO (redactar):_ quién usa el pronóstico (jefe de operación / programación de flota), qué decisión (despachos, vehículos y conductores por ruta y día), con cuánta anticipación (30 días) y con qué información (histórico hasta el día t y calendario conocido). Ninguna variable usada depende de información posterior a t.

## 2. Descripción del conjunto y diccionario

In [ ]:
ID = cfg["datos"]["dataset_id"]
FECHAS = m1.filtro_fechas(cfg["datos"]["fecha_inicio"], cfg["datos"]["fecha_fin"])

resumen = m1.soql(ID, {"$select": "count(*) as registros, min(fecha_despacho) as desde, max(fecha_despacho) as hasta"})
resumen_valido = m1.soql(ID, {"$select": "count(*) as registros, min(fecha_despacho) as desde, max(fecha_despacho) as hasta", "$where": FECHAS})
display(pd.concat({"todo": resumen, "rango válido": resumen_valido}))

In [ ]:
import requests
meta = requests.get(f"https://www.datos.gov.co/api/views/{ID}.json", timeout=60).json()
diccionario = pd.DataFrame([{"variable": c["fieldName"], "tipo": c["dataTypeName"], "descripcion": c.get("description", "")}
                            for c in meta["columns"]])
diccionario.to_csv(DATA / "external" / "diccionario_eh75-8ah6.csv", index=False)
pd.set_option("display.max_colwidth", 200)
diccionario

In [ ]:
# Valores reales de las variables categóricas, para contrastar con el diccionario
for col in ["tipo_despacho", "clase_vehiculo", "nivel_servicio"]:
    display(m1.soql(ID, {"$select": f"{col}, count(*) as n", "$where": FECHAS, "$group": col, "$order": "n DESC"}))

**Inconsistencias diccionario vs. datos** (verificar y completar):
- `fecha_despacho`: hay fechas imposibles (máximo en el año 6202) → se acota a `fecha_inicio..fecha_fin`.
- `tipo_despacho`: el diccionario dice códigos `1`/`2`; los datos traen `ORIGEN`/`TRANSITO`.
- `clase_vehiculo`: el diccionario dice "identificador numérico RUNT"; los datos traen texto (BUS, MICROBUS…).
- `nivel_servicio`: el diccionario lista códigos 1–3; los datos traen texto.
- `municipio_destino_ruta`: el diccionario dice "código DIVIPOLA del **departamento**", pero los valores tienen 5 dígitos (municipio).

## 3. Selección de rutas

Se toman las `n_rutas` rutas (origen→destino, despacho en origen) con más pasajeros y, si `incluir_regreso`, su sentido de regreso.

In [ ]:
ruta_top = DATA / "raw" / "top_rutas.csv"
if FORZAR_DESCARGA or not ruta_top.exists():
    m1.rutas_principales(ID, 50, FECHAS).to_csv(ruta_top, index=False)
top = pd.read_csv(ruta_top, dtype={"municipio_origen_ruta": str, "municipio_destino_ruta": str})
top.head(15)

## 4. DIVIPOLA

Codificación oficial del DANE. _TODO: guardar el archivo oficial en `data/external/divipola.csv` (columnas `cod_mpio`, `nom_mpio`, `nom_dpto`) y citar la fuente._

In [ ]:
ruta_divipola = DATA / "external" / "divipola.csv"
if ruta_divipola.exists():
    divipola = pd.read_csv(ruta_divipola, dtype=str).set_index("cod_mpio")
    nombre = lambda c: divipola["nom_mpio"].get(c, c)
else:
    print("⚠ Falta data/external/divipola.csv — se usan códigos")
    nombre = lambda c: c
top["origen"] = top.municipio_origen_ruta.map(nombre)
top["destino"] = top.municipio_destino_ruta.map(nombre)
top[["origen", "destino", "pasajeros", "despachos", "desde", "hasta"]].head(15)

In [ ]:
n = cfg["datos"]["n_rutas"]
sel = top.head(n)[["municipio_origen_ruta", "municipio_destino_ruta"]].values.tolist()
pares = [tuple(p) for p in sel]
if cfg["datos"]["incluir_regreso"]:
    pares += [(d, o) for o, d in sel]
pares = list(dict.fromkeys(pares))

rutas = pd.DataFrame(pares, columns=["origen", "destino"])
rutas["ruta"] = rutas.origen + "-" + rutas.destino
rutas["sentido"] = ["ida"] * len(sel) + ["regreso"] * (len(rutas) - len(sel))
rutas["ruta_par"] = [f"{o}-{d}" if s == "ida" else f"{d}-{o}" for o, d, s in zip(rutas.origen, rutas.destino, rutas.sentido)]
rutas["nombre"] = rutas.origen.map(nombre) + " → " + rutas.destino.map(nombre)
rutas

## 5. Descarga agregada por ruta, día y clase de vehículo

In [ ]:
ruta_diario = DATA / "raw" / "diario_rutas.parquet"
ruta_dias = DATA / "raw" / "dias_con_reporte.parquet"
if FORZAR_DESCARGA or not ruta_diario.exists():
    m1.diario_por_ruta(ID, pares, FECHAS).to_parquet(ruta_diario)
    pd.Series(m1.dias_con_reporte(ID, FECHAS), name="fecha").to_frame().to_parquet(ruta_dias)
diario = pd.read_parquet(ruta_diario)
dias_validos = pd.DatetimeIndex(pd.read_parquet(ruta_dias)["fecha"])
print(diario.shape); diario.head()

## 6. Calidad: despachos con cero pasajeros, días sin reporte, atípicos

In [ ]:
calidad = diario.assign(cero=(diario.despachos > 0) & (diario.pasajeros == 0),
                        ocup=diario.pasajeros / diario.despachos.where(diario.despachos > 0))
resumen_calidad = calidad.groupby("ruta").agg(registros=("cero", "size"), despachos_cero_pasajeros=("cero", "sum"),
                                              pasajeros_por_despacho_mediana=("ocup", "median"),
                                              pasajeros_por_despacho_p99=("ocup", lambda s: s.quantile(.99)))
resumen_calidad

In [ ]:
# Pasajeros por despacho imposibles dado el tipo de vehículo → anómalos (umbral = 1.5 × capacidad configurada)
cap = {k.upper(): v for k, v in cfg["operacion"]["capacidad_por_clase"].items()}
lim = diario.clase_vehiculo.map(cap).fillna(max(cap.values())) * 1.5
diario["anomalo_ocupacion"] = (diario.pasajeros / diario.despachos.clip(lower=1)) > lim
print("Registros con ocupación imposible:", int(diario.anomalo_ocupacion.sum()))

In [ ]:
# Serie objetivo: pasajeros por ruta y día (los registros con ocupación imposible se excluyen)
serie = (diario[~diario.anomalo_ocupacion].groupby(["fecha", "ruta"]).pasajeros.sum().unstack("ruta"))
serie = m1.completar_calendario(serie, dias_validos)

atipicos = serie.apply(m1.marcar_atipicos)
sin_reporte = serie.isna()
print("Días sin reporte del sistema:", int(sin_reporte.iloc[:, 0].sum()))
print("Atípicos por ruta:\n", atipicos.sum())

## 7. Festivos, imputación y exportación

Atípicos en festivos y días adyacentes NO se imputan (son demanda real). Calendario:  (variable exógena)

Fuente primaria: Ley 51 de 1983 (traslado de festivos al lunes) y calendario oficial. Se genera con el paquete `holidays` y se guarda en `data/external/` para no depender de internet. _TODO: verificar contra la fuente primaria y citarla._

In [ ]:
import holidays
anios = range(serie.index.min().year, serie.index.max().year + 2)
co = holidays.Colombia(years=anios, language="es")
fest = pd.DataFrame(sorted(co.items()), columns=["fecha", "nombre"])
fest["fecha"] = pd.to_datetime(fest.fecha)
fest.to_csv(DATA / "external" / "festivos_colombia.csv", index=False)
fest.tail()

In [ ]:
es_festivo = serie.index.isin(fest.fecha) | serie.index.isin(fest.fecha - pd.Timedelta(days=1)) | serie.index.isin(fest.fecha + pd.Timedelta(days=1))
a_imputar = sin_reporte | (atipicos & ~es_festivo[:, None])
serie_limpia = serie.mask(a_imputar).apply(m1.imputar_estacional)

calendario = m1.tabla_calendario(pd.date_range(serie.index.min(), serie.index.max() + pd.Timedelta(days=H), freq="D"), fest)

(DATA / "processed").mkdir(exist_ok=True)
serie_limpia.astype("float32").to_parquet(DATA / "processed" / "series_diarias.parquet")
serie.to_parquet(DATA / "processed" / "series_diarias_sin_imputar.parquet")
calendario.to_parquet(DATA / "processed" / "calendario.parquet")
rutas.to_csv(DATA / "processed" / "rutas.csv", index=False)
resumen_calidad.assign(dias_imputados=a_imputar.sum()).to_csv(DATA / "processed" / "calidad.csv")
serie_limpia.describe().T

In [ ]:
fig, axes = plt.subplots(len(rutas), 1, figsize=(12, 2.2 * len(rutas)), sharex=True)
for ax, (_, r) in zip(np.atleast_1d(axes), rutas.iterrows()):
    ax.plot(serie_limpia.index, serie_limpia[r.ruta], lw=.6)
    ax.scatter(serie.index[a_imputar[r.ruta]], serie_limpia.loc[a_imputar[r.ruta], r.ruta], s=4, c="C3", label="imputado")
    ax.set_title(r.nombre, fontsize=9)
axes[0].legend()
fig.suptitle("Figura 1. Pasajeros diarios por ruta (despacho en origen)")
fig.tight_layout(); fig.savefig(FIG / "fig01_series_rutas.png", dpi=150)